## Read in TROPESS

In [ ]:
import sys
import os
import numpy as np
import xarray as xr
import pandas as pd
import datetime
import matplotlib as mpl
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


In [ ]:
xls = []
folder = '/Volumes/ESA_F4R/tropess/AIRS-Aqua/reanalysis/'
arr = os.listdir(folder)
print(arr)
for I,F in enumerate(arr):
    try:
        daynight = x = xr.open_dataset(folder+F,engine="netcdf4", group='geophysical')['day_night_flag'] 
        x = xr.open_dataset(folder+F,
                            engine="netcdf4",
                            drop_variables=['x_h2o_col_p_error','x_h2o_col_p','dd_004','dd_col_p',
                                            'dd_col_p_error','datetime_utc','datetime_utc_dim', 
                                            'microwindow','microwindow_range',
                                            'altitude','level','target_idx','target_id','year_fraction'])
        x = x.where(daynight==1,drop=True)
        x = x.rename({'latitude':'lat','longitude':'lon','x_h2o':'H2O','x':'ratio'})
        #print(x['ratio'].min().values)
        #x['ratio'] = x['ratio'].where(x['ratio']!=-999.,drop=True)
        #x['H2O'] = x['H2O'].where(x['H2O']!=-999.,drop=True)
        #print(x['ratio'].min().values)
        pdx = x.to_dataframe()
        pdx = pdx.reset_index()
        #print(pdx)
        xls.append(pdx)
        x.close()
    except:
        print('not nc4: ',F)

#folder = '/Volumes/ESA_F4R/tropess/AIRS-Aqua/forward/central_africa/'
#arr = os.listdir(folder)
#print(arr)
#for I,F in enumerate(arr):
#    try:
#        daynight = x = xr.open_dataset(folder+F,engine="netcdf4", group='geophysical')['day_night_flag'] 
#        x = xr.open_dataset(folder+F,
#                            engine="netcdf4",
#                            drop_variables=['x_h2o_col_p_error','x_h2o_col_p','dd_004','dd_col_p',
#                                            'dd_col_p_error','datetime_utc','datetime_utc_dim', 
#                                            'microwindow','microwindow_range',
#                                            'altitude','level','target_idx','target_id','year_fraction'])
#        x = x.where(daynight==1,drop=True)
#        x = x.rename({'latitude':'lat','longitude':'lon','x_h2o':'H2O','x':'ratio'})
#        pdx = x.to_dataframe()
#        pdx = pdx.reset_index()
#        #print(pdx)
#        xls.append(pdx)
#        x.close()
#    except:
#        print('not nc4: ',F)
#

In [ ]:
ds = pd.concat(xls,ignore_index=True)
print(ds)
ds = ds.reset_index()

In [ ]:
# calculate deltaD
R = ds.ratio
# use this RSMOW for molecules of water
Rvsmow = 3.11*10**(-4)
dD = (R/Rvsmow - 1)*1000.
ds['deltaD'] = dD

ds = ds.dropna()

#integer the pressure
ds['pressure'] = ds['pressure'].astype(int)

In [ ]:

print(ds['deltaD'].loc[(ds.pressure<830)&(ds.pressure>820)].describe())


In [ ]:
import cartopy.crs as ccrs
import cartopy.feature
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
ax.add_feature(cartopy.feature.OCEAN)
dsm = ds.loc[(ds.pressure>820)&(ds.pressure<830)]
dsmp = dsm.groupby(['lat','lon']).mean()
dsmd = dsmp['deltaD']
dsmd = dsmd.reset_index()
print(dsmd)
dsmd.plot(x='lon',y='lat',kind='scatter',c='deltaD',
    ax=ax,transform=ccrs.PlateCarree(),
    vmin=-400,vmax=0,
    alpha=1,
    cmap=plt.cm.viridis,
    )

#ax.set_extent([-20, 20, -20, 60])
ax.set_extent([7, 32, -15, 12])
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                  linewidth=1, color='black', alpha=0.5, linestyle='dotted')
gl.top_labels = False
gl.right_labels = False
#ax.set_title()
#plt.savefig()
plt.show()
plt.clf()

In [ ]:
cb_tropess = ds.loc[(ds.lat>-5)&(ds.lat<5)&(ds.lon>8)&(ds.lon<29)]

cb_tropess = cb_tropess.set_index(['time'])
cb_tropess = cb_tropess.sort_index()
cb_tropess = cb_tropess.drop(columns=['level','target','ratio'])
cb_xr = cb_tropess.to_xarray()
print(cb_xr)

In [ ]:

cb_xr.to_netcdf(path='/Users/ellendyer/Documents/GitHub/Isotopes_F4R/iso_prepped/tropess_airs_eq_cb.nc',mode='w',format='NETCDF4',engine='netcdf4')